# Custom Words LipNet Training

Loads data/custom_words/manifest.csv, resolves each relative .npz path, pads clips to a shared length, creates an 80/20 stratified split with seed 42, and trains models.valente.model.LipNet with CTC.

In [14]:
# !pip install torch numpy pandas matplotlib

In [15]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import sys
from pathlib import Path

import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from torch.utils.data import DataLoader, Dataset

# Import from the repository root so the shared transformer blocks resolve.
PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from models.valente.model import build_lipnet, ctc_greedy_decode, tokens_to_text


In [16]:
MANIFEST_PATH = Path('../../data/custom_words/manifest.csv')
RANDOM_SEED, TEST_FRACTION = 42, 0.20
manifest = pd.read_csv(MANIFEST_PATH)
if not {'relative_path', 'label'}.issubset(manifest.columns):
    raise ValueError('manifest.csv must contain relative_path and label columns.')

manifest_dir = MANIFEST_PATH.parent.resolve()
manifest['npz_path'] = manifest['relative_path'].map(lambda path: (manifest_dir / path).resolve())
missing_paths = manifest.loc[~manifest['npz_path'].map(Path.is_file), 'npz_path']
if not missing_paths.empty:
    raise FileNotFoundError(f'Missing .npz file: {missing_paths.iloc[0]}')
print(f'Samples: {len(manifest)} | Labels: {sorted(manifest.label.unique())}')

Samples: 320 | Labels: ['bad', 'bye', 'good', 'hello', 'hi', 'no', 'thanks', 'yes']


In [17]:
def load_features_and_labels(manifest_dataframe):
    clips, frame_shape = [], None
    for row in manifest_dataframe.itertuples(index=False):
        with np.load(row.npz_path, allow_pickle=False) as archive:
            if 'frames' not in archive:
                raise KeyError(f'Missing frames array in {row.npz_path}')
            clip = archive['frames']
        if frame_shape is None:
            frame_shape = clip.shape[1:]
        elif clip.shape[1:] != frame_shape:
            raise ValueError(f'Inconsistent frame size in {row.npz_path.name}: {clip.shape[1:]}')
        clips.append(clip)

    frame_lengths = np.array([clip.shape[0] for clip in clips], dtype=np.int64)
    features = np.zeros((len(clips), int(frame_lengths.max()), *frame_shape), dtype=clips[0].dtype)
    for index, clip in enumerate(clips):
        features[index, :clip.shape[0]] = clip
    labels = manifest_dataframe['label'].to_numpy(dtype=str)
    return features, labels, frame_lengths

features, labels, frame_lengths = load_features_and_labels(manifest)
print('features:', features.shape, features.dtype)
print('labels:', labels.shape, labels.dtype)
print('frame lengths:', frame_lengths.min(), 'to', frame_lengths.max())

features: (320, 92, 50, 100) uint8
labels: (320,) <U6
frame lengths: 89 to 92


In [18]:
def stratified_split(labels, test_fraction=0.20, seed=42):
    rng = np.random.default_rng(seed)
    train_parts, test_parts = [], []
    for label in np.unique(labels):
        indices = np.flatnonzero(labels == label)
        rng.shuffle(indices)
        test_count = max(1, round(len(indices) * test_fraction))
        test_parts.append(indices[:test_count])
        train_parts.append(indices[test_count:])
    train_indices, test_indices = np.concatenate(train_parts), np.concatenate(test_parts)
    rng.shuffle(train_indices)
    rng.shuffle(test_indices)
    return train_indices, test_indices

train_indices, test_indices = stratified_split(labels, TEST_FRACTION, RANDOM_SEED)
X_train, X_test = features[train_indices], features[test_indices]
y_train, y_test = labels[train_indices], labels[test_indices]
train_frame_lengths, test_frame_lengths = frame_lengths[train_indices], frame_lengths[test_indices]

class_names = np.unique(labels)
label_to_index = {label: index for index, label in enumerate(class_names)}
y_train_encoded = np.array([label_to_index[label] for label in y_train], dtype=np.int64)
y_test_encoded = np.array([label_to_index[label] for label in y_test], dtype=np.int64)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Class mapping:', label_to_index)

X_train: (256, 92, 50, 100) | X_test: (64, 92, 50, 100)
Class mapping: {np.str_('bad'): 0, np.str_('bye'): 1, np.str_('good'): 2, np.str_('hello'): 3, np.str_('hi'): 4, np.str_('no'): 5, np.str_('thanks'): 6, np.str_('yes'): 7}


## Recommended Settings

Start with 60 epochs, a learning rate of 1e-4, and batch size 2. This is a small 256-clip training set and LipNet is a memory-intensive 3D-CNN plus BiGRU model. Early stopping and validation-loss learning-rate decay are included below.

In [19]:
EPOCHS = 200
LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-6
BATCH_SIZE = 1
PATIENCE = 25
CHECKPOINT_PATH = Path('../../checkpoints/valente_custom_words_lipnet_transformer.pt')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Temporal core: 'transformer' replaces the two BiGRUs ('bigru' reproduces the original).
ARCHITECTURE = 'transformer'
MODEL_SETTINGS = {
    'architecture': ARCHITECTURE,
    # Width matches the BiGRU baseline (2 x 256 bidirectional = 512) so the
    # comparison tests attention versus recurrence rather than raw capacity.
    'hidden_size': 256,
    'dropout': 0.5,
    'transformer_layers': 3,
    'transformer_heads': 4,
    'transformer_feedforward': 512,
}

characters = sorted(set(''.join(labels)))
char_to_index = {character: index + 1 for index, character in enumerate(characters)}
index_to_char = {index: character for character, index in char_to_index.items()}
CTC_BLANK = 0
print('Device:', device, '| Character vocabulary:', characters)
print('Architecture:', ARCHITECTURE)


Device: cuda | Character vocabulary: ['a', 'b', 'd', 'e', 'g', 'h', 'i', 'k', 'l', 'n', 'o', 's', 't', 'y']


In [20]:
class WordClipDataset(Dataset):
    def __init__(self, features, labels, frame_lengths):
        self.features, self.labels, self.frame_lengths = features, labels, frame_lengths

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        length = int(self.frame_lengths[index])
        real_frames = self.features[index, :length].astype(np.float32) / 255.0
        real_frames = (real_frames - real_frames.mean()) / (real_frames.std() + 1e-6)
        # Keep padding at zero so the model can use the supplied true length.
        clip = np.zeros_like(self.features[index], dtype=np.float32)
        clip[:length] = real_frames
        target = torch.tensor([char_to_index[char] for char in self.labels[index]], dtype=torch.long)
        return torch.from_numpy(clip).unsqueeze(1), target, length, self.labels[index]

def ctc_collate(batch):
    clips, targets, lengths, words = zip(*batch)
    return torch.stack(clips), torch.cat(targets), torch.tensor(lengths), torch.tensor([len(target) for target in targets]), list(words)

train_loader = DataLoader(WordClipDataset(X_train, y_train, train_frame_lengths), batch_size=BATCH_SIZE, shuffle=True, collate_fn=ctc_collate)
test_loader = DataLoader(WordClipDataset(X_test, y_test, test_frame_lengths), batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate)
print('One batch:', next(iter(train_loader))[0].shape)

One batch: torch.Size([1, 92, 1, 50, 100])


In [21]:
model = build_lipnet(MODEL_SETTINGS, num_classes=len(char_to_index) + 1, input_channels=1).to(device)
# feature_projection is an nn.LazyLinear: its shape is only known after the
# first forward pass, so the parameter count is reported after training starts.
criterion = nn.CTCLoss(blank=CTC_BLANK, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
# Smoothly anneal from 1e-4 to 1e-6 across all configured epochs.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=MIN_LEARNING_RATE,
)

def decode_batch(log_probs, input_lengths):
    tokens = ctc_greedy_decode(log_probs.detach().cpu(), input_lengths.cpu(), CTC_BLANK)
    return [tokens_to_text(sequence, index_to_char) for sequence in tokens]

def run_epoch(loader, training):
    model.train(training)
    total_loss, total_exact, total_samples = 0.0, 0, 0
    for clips, targets, input_lengths, target_lengths, words in loader:
        clips, targets = clips.to(device), targets.to(device)
        with torch.set_grad_enabled(training):
            log_probs, output_lengths = model.ctc_log_probs(clips, input_lengths)
            loss = criterion(log_probs, targets, output_lengths.cpu(), target_lengths.cpu())
            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
        predictions = decode_batch(log_probs, output_lengths)
        total_loss += loss.item() * len(words)
        total_exact += sum(prediction == target for prediction, target in zip(predictions, words))
        total_samples += len(words)
    return total_loss / total_samples, total_exact / total_samples

In [ ]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
best_validation_loss, stale_epochs = float('inf'), 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    validation_loss, validation_accuracy = run_epoch(test_loader, training=False)
    scheduler.step()
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss {train_loss:.4f}, accuracy {train_accuracy:.3f} | validation loss {validation_loss:.4f}, accuracy {validation_accuracy:.3f} | lr {optimizer.param_groups[0]["lr"]:.1e}')

    if validation_loss < best_validation_loss:
        best_validation_loss, stale_epochs = validation_loss, 0
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'char_to_index': char_to_index, 'validation_loss': validation_loss, **MODEL_SETTINGS}, CHECKPOINT_PATH)
    else:
        stale_epochs += 1
        if stale_epochs >= PATIENCE:
            print(f'Early stopping after {PATIENCE} epochs without improvement.')
            # break

print('Best checkpoint:', CHECKPOINT_PATH)

In [23]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
print("Best checkpoint epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["validation_loss"])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
clips, _, input_lengths, _, words = next(iter(test_loader))
with torch.no_grad():
    log_probs, output_lengths = model.ctc_log_probs(clips.to(device), input_lengths)
for target, prediction in zip(words[:8], decode_batch(log_probs, output_lengths)[:8]):
    print(f'Target: {target:7s} | Prediction: {prediction}')

Best checkpoint epoch: 171
Validation loss: 0.5019979945282103
Target: thanks  | Prediction: yeaks


In [24]:
# Press q in the webcam window to stop recording and predict the captured word.
import cv2
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve().parents[1]  # models/valente -> VisionSpeak
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from processing.preprocessor import Preprocessor
from processing.video_loader import VideoLoader

MAX_TRAINING_FRAMES = 92
loaded_char_to_index = checkpoint['char_to_index']
loaded_index_to_char = {index: char for char, index in loaded_char_to_index.items()}

def prepare_webcam_clip(frames, max_frames):
    if len(frames) == 0:
        raise ValueError('No webcam frames were captured.')
    if len(frames) > max_frames:
        indices = np.linspace(0, len(frames) - 1, max_frames).round().astype(int)
        frames = frames[indices]

    length = len(frames)
    real_frames = frames.astype(np.float32) / 255.0
    real_frames = (real_frames - real_frames.mean()) / (real_frames.std() + 1e-6)
    padded_frames = np.zeros((max_frames, *real_frames.shape[1:]), dtype=np.float32)
    padded_frames[:length] = real_frames
    tensor = torch.from_numpy(padded_frames).unsqueeze(0).unsqueeze(2).to(device)
    return tensor, torch.tensor([length], dtype=torch.long)

loader = VideoLoader()
preprocessor = Preprocessor()
print('Webcam is open. Speak one trained word, then press q to predict.')
video_data = loader.load('webcam', show_full_frame=True)
capture = video_data.get('capture')
if capture is not None:
    capture.release()
cv2.destroyAllWindows()

video_data = preprocessor.convert_to_greyscale(video_data)
webcam_frames = np.asarray(video_data['frames'], dtype=np.uint8)
webcam_tensor, webcam_lengths = prepare_webcam_clip(webcam_frames, MAX_TRAINING_FRAMES)

with torch.no_grad():
    webcam_log_probs, webcam_output_lengths = model.ctc_log_probs(webcam_tensor, webcam_lengths)
webcam_tokens = ctc_greedy_decode(webcam_log_probs.cpu(), webcam_output_lengths.cpu(), CTC_BLANK)[0]
webcam_prediction = tokens_to_text(webcam_tokens, loaded_index_to_char)
print(f'Captured frames: {len(webcam_frames)}')
print(f'Prediction: {webcam_prediction}')

Webcam is open. Speak one trained word, then press q to predict.
Captured frames: 112
Prediction: hello
